# Deconvolve Positive Control Genes and plot
February 8, 2024

This notebook is designed to simplify the deconvolution process for gene expression and chromatin in one pass. The functions save the output to disk as plots, and eventually as data and metadata for downstream analysis.


In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
def deconvolve_gene(gene_name, replicate, save_dir):
    """
    Deconvolve a gene's gene expression and chromatin
    """
    from src.model import Model
    from cc_src.chromatin_model import ChromatinModel
    from src.config import load_yl_replicate1_rg1_alpha_vst_config, \
        load_yl_replicate2_rg1_alpha_vst_config
    
    if replicate == 1:
        config = load_yl_replicate1_rg1_alpha_vst_config()
    elif replicate == 2:
        config = load_yl_replicate2_rg1_alpha_vst_config()
    else:
        raise ValueError("Undefined replicate")
    
    print(f"Deconvolving {gene_name}, replicate={replicate}")
    print(f"Deconvolving gene expression...", end="")
    ge_model = Model(config, gene_name)
    ge_model.deconvolve_find_optimal_gamma()
    print("Done.")

    print(f"Deconvolving chromatin...", end="")
    chrom_model = ChromatinModel(config)
    chrom_model.load_mnase_gene(gene_name, replicate=config.replicate)
    chrom_model.deconvolve_find_optimal_gamma()
    print("Done.")
    
    save_name = f"{save_dir}/{gene_name}_rep{config.replicate}_chromatin.png"
    fig = chrom_model.create_deconvolution_plots_abbreviated_flipped(ge_model=ge_model)
    plt.savefig(save_name, dpi=200)
    print(f"Saved figure to {save_name}")
    plt.close(fig)

    save_name = f"{save_dir}/{gene_name}_{config.replicate}_predicted.png"
    fig = chrom_model.plot_prediction_comparison()
    plt.savefig(save_name, dpi=200)
    print(f"Saved figure to {save_name}")
    plt.close(fig)


In [ ]:
from cc_src.geneset import positive_control_genes
from src.timer import Timer

save_dir1 = "output/positive_controls_chromatin/rep1"
save_dir2 = "output/positive_controls_chromatin/rep2"

save_dirs = {
    1: save_dir1,
    2: save_dir2
}

# For each replicate, deconvolve the positive control genes
# and save the resulting figures to disk
# TODO: Save the deconvolved data and metadata to disk
#       Can use existing code for this, but requires some cleanup
#       and simplification
timer = Timer()

genes = positive_control_genes()
total = len(genes)*2
count = 0
for replicate in [1, 2]:
    save_dir = save_dirs[replicate]
    for gene_name in genes:
        deconvolve_gene(gene_name, replicate, save_dir)
        count += 1
        print(f"\n===== Progress {count}/{total} - {timer.get_time()}\n\n")


Deconvolving CLB2, replicate=1
Deconvolving gene expression...Done.
Deconvolving chromatin...Loading MNase reads for CLB2...Done.
The histogram shape around the TSS is: (3, 9)
The shape of the flattened grid to be deconvolved is: (16, 27)
Applying normalization using scaling matrix: output/mnase/rep1_len_scaling_3len_bins.csv
The shape of the flattened grid to be deconvolved is: (16, 27)
Running the find optimal gamma procedure...
  ... The base fitting norm (rn) with no smoothing (gamma=0) is: 0.8319
  ... Searching for an optimal gamma value in the boundaries: [0.0001, 0.0100]
  ...  search left, rn_goal = 0.9119, rate = 9.6
  ...   gm = 0.0050, rn = 1.0323, rate = 24.09, time = 00:00:34.63
  ...   gm = 0.0026, rn = 0.9649, rate = 15.99, time = 00:00:43.29
  ...   gm = 0.0013, rn = 0.9233, rate = 10.99, time = 00:00:52.15
  ...  search right, rn_goal = 1.1647, rate = 40.0
  ... rn range: [0.9233, 1.1355]
  ... search gamma in [0.0013 0.0100] for elbow
  ...   gm = 0.0013, rn = 0.9233

/Users/trung/opt/anaconda3/envs/cell-cycle-deconvolution/lib/python3.8/site-packages/cvxpy/problems/problem.py:1403: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


  ...   gm = 0.0050, rn = 0.7823, rate = 29.34, time = 00:01:43.91
  ...   gm = 0.0026, rn = 0.7161, rate = 18.38, time = 00:02:28.96
  ...   gm = 0.0013, rn = 0.6793, rate = 12.31, time = 00:03:13.50
  ...  search right, rn_goal = 0.9249, rate = 52.9
  ... rn range: [0.6793, 0.8832]
  ... search gamma in [0.0013 0.0100] for elbow
  ...   gm = 0.0013, rn = 0.6793, rate = 12.31, time = 00:04:40.37
  ...   gm = 0.0022, rn = 0.7073, rate = 16.92, time = 00:05:24.46
  ...   gm = 0.0031, rn = 0.7309, rate = 20.84, time = 00:06:07.97
  ...   gm = 0.0039, rn = 0.7543, rate = 24.70, time = 00:06:50.72
  ...   gm = 0.0048, rn = 0.7768, rate = 28.42, time = 00:07:33.35
  ...   gm = 0.0057, rn = 0.7965, rate = 31.68, time = 00:08:16.92
  ...   gm = 0.0065, rn = 0.8155, rate = 34.81, time = 00:08:57.93
  ...   gm = 0.0074, rn = 0.8366, rate = 38.31, time = 00:09:41.58
  ...   gm = 0.0083, rn = 0.8539, rate = 41.17, time = 00:10:22.34
  ...   gm = 0.0091, rn = 0.8697, rate = 43.78, time = 00:11:07.

/Users/trung/opt/anaconda3/envs/cell-cycle-deconvolution/lib/python3.8/site-packages/cvxpy/problems/problem.py:1403: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


  ...   gm = 0.0050, rn = 2.3292, rate = 6.49, time = 00:01:38.69
  ...   gm = 0.0026, rn = 2.2921, rate = 4.80, time = 00:02:21.38
  ...   gm = 0.0013, rn = 2.2583, rate = 3.25, time = 00:03:04.82
  ...  search right, rn_goal = 3.0621, rate = 40.0
  ... rn range: [2.2583, 2.3931]
  ... search gamma in [0.0013 0.0100] for elbow
  ...   gm = 0.0013, rn = 2.2583, rate = 3.25, time = 00:04:27.53
  ...   gm = 0.0022, rn = 2.2848, rate = 4.46, time = 00:05:10.36
  ...   gm = 0.0031, rn = 2.3006, rate = 5.18, time = 00:05:51.18
  ...   gm = 0.0039, rn = 2.3151, rate = 5.85, time = 00:06:30.48
  ...   gm = 0.0048, rn = 2.3265, rate = 6.37, time = 00:07:10.34
  ...   gm = 0.0057, rn = 2.3371, rate = 6.85, time = 00:07:51.79
  ...   gm = 0.0065, rn = 2.3503, rate = 7.46, time = 00:08:30.80
  ...   gm = 0.0074, rn = 2.3649, rate = 8.13, time = 00:09:10.18
  ...   gm = 0.0083, rn = 2.3757, rate = 8.62, time = 00:09:49.51
  ...   gm = 0.0091, rn = 2.3836, rate = 8.98, time = 00:10:29.12
  ...   gm

...

  ...   gm = 0.0057, rn = 0.6125, rate = 27.30, time = 00:02:05.24
  ...   gm = 0.0065, rn = 0.6225, rate = 29.39, time = 00:02:14.96
  ...   gm = 0.0074, rn = 0.6311, rate = 31.17, time = 00:02:24.79
  ...   gm = 0.0083, rn = 0.6387, rate = 32.75, time = 00:02:34.84
  ...   gm = 0.0091, rn = 0.6463, rate = 34.33, time = 00:02:44.90
  ...   gm = 0.0100, rn = 0.6538, rate = 35.89, time = 00:02:54.43
The x_grad1 is: [0.01657436 0.01555071 0.01377595 0.01185037 0.0109507  0.01064778
 0.00932095 0.00808124 0.00759525 0.00755198 0.00750409]
The x_grad2 is: [-1.02365332e-03 -1.39920782e-03 -1.85017168e-03 -1.41262389e-03
 -6.01290779e-04 -8.14873610e-04 -1.28327138e-03 -8.62848233e-04
 -2.64632115e-04 -4.55840575e-05 -4.78900012e-05]
The curvature is: [ 0.00012219  0.00029012  0.00076109  0.00056894  0.00126612  0.0021179
  0.00266826  0.00288515  0.00384726  0.00084609 -0.00043658]
YNR067C: ... final gamma = 0.00827
Time to find optimal gamma: 00:03:04.44
Deconvolved in : 00:03:05.08
The fi

/Users/trung/opt/anaconda3/envs/cell-cycle-deconvolution/lib/python3.8/site-packages/cvxpy/problems/problem.py:1403: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


  ...   gm = 0.0050, rn = 0.8598, rate = 28.73, time = 00:01:44.43
  ...   gm = 0.0026, rn = 0.7889, rate = 18.11, time = 00:02:29.71
  ...   gm = 0.0013, rn = 0.7472, rate = 11.87, time = 00:03:14.87
  ...  search right, rn_goal = 0.9879, rate = 47.9
  ... rn range: [0.7472, 0.9579]
  ... search gamma in [0.0013 0.0100] for elbow
  ...   gm = 0.0013, rn = 0.7472, rate = 11.87, time = 00:04:44.64
  ...   gm = 0.0022, rn = 0.7775, rate = 16.40, time = 00:05:29.98
  ...   gm = 0.0031, rn = 0.8037, rate = 20.33, time = 00:06:15.07
  ...   gm = 0.0039, rn = 0.8298, rate = 24.23, time = 00:06:59.90
  ...   gm = 0.0048, rn = 0.8534, rate = 27.77, time = 00:07:44.41
  ...   gm = 0.0057, rn = 0.8775, rate = 31.37, time = 00:08:29.17
  ...   gm = 0.0065, rn = 0.8996, rate = 34.68, time = 00:09:13.63
  ...   gm = 0.0074, rn = 0.9159, rate = 37.12, time = 00:09:57.12
  ...   gm = 0.0083, rn = 0.9308, rate = 39.35, time = 00:10:40.74
  ...   gm = 0.0091, rn = 0.9440, rate = 41.33, time = 00:11:24.

/Users/trung/opt/anaconda3/envs/cell-cycle-deconvolution/lib/python3.8/site-packages/cvxpy/problems/problem.py:1403: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


  ...   gm = 0.0050, rn = 2.3949, rate = 6.08, time = 00:01:44.54
